# BidMate-DocAgent — 5-minute reviewer quickstart

This notebook gets the RFP-focused RAG pipeline running on Google Colab with **no local setup**. Five sequential cells:

1. Clone the public repo.
2. Install dependencies.
3. Build the public synthetic RFP index.
4. Ask a single grounded question with the extractive baseline.
5. Inspect the structured answer (`schema_version: 2`).

**Out of scope here**: the live Streamlit UI (see [`docs/deployment.md`](https://github.com/hskim-solv/BidMate-DocAgent/blob/main/docs/deployment.md)), the LLM-synthesis preset (requires `ANTHROPIC_API_KEY`), and the private 100-doc real-data eval. This notebook uses the hashing backend (offline, free, deterministic).

Total runtime: ~2–3 minutes on a free Colab T4 (mostly the `pip install`).

## 1. Clone the repo

In [ ]:
!git clone https://github.com/hskim-solv/BidMate-DocAgent.git
%cd BidMate-DocAgent

## 2. Install dependencies

The `--quiet` flag is just to keep the cell output readable. Heavy deps (`sentence-transformers`, `pymupdf`, `pytesseract`) install in ~90 s on Colab.

In [ ]:
!pip install -r requirements.txt --quiet

## 3. Build the public synthetic RFP index

Seven synthetic RFPs (`data/raw/rfp_agency_*.json`) → `data/index/index.json` in <10 s on Colab. The hashing backend is the default (`EMBEDDING_BACKEND` unset).

In [ ]:
!python scripts/build_index.py --input_dir data/raw --output_dir data/index

## 4. Ask a question with the extractive baseline

`app.py` runs the pipeline once and writes the answer JSON to `outputs/answer.json`. The default pipeline is `naive_baseline` (ADR 0001); pass `--pipeline agentic_full` to exercise metadata-first + verifier/retry.

In [ ]:
!python app.py --input_dir data/index --output_dir outputs --query "기관 A의 AI 품질관리 요구사항은 무엇인가?"

## 5. Inspect the structured answer

Per [ADR 0003](https://github.com/hskim-solv/BidMate-DocAgent/blob/main/docs/adr/0003-structured-answer-citation-contract.md), every answer is a `schema_version: 2` JSON with `status`, `claims[].citations[]`, and a top-level `evidence` list. Each `chunk_id` resolves into `evidence` — no hallucinated citations by construction (see [`docs/answer-policy.md`](https://github.com/hskim-solv/BidMate-DocAgent/blob/main/docs/answer-policy.md)).

In [ ]:
import json

with open("outputs/answer.json", encoding="utf-8") as f:
    answer = json.load(f)

print("status:", answer.get("answer", {}).get("status"))
print()
print("summary:", answer.get("answer", {}).get("summary"))
print()
print("claims (", len(answer.get("answer", {}).get("claims", [])), "):")
for claim in answer.get("answer", {}).get("claims", []):
    cite_ids = [c.get("chunk_id") for c in claim.get("citations", [])]
    print(f"  - {claim.get('target')}: {claim.get('claim')[:120]}")
    print(f"    citations: {cite_ids}")

## Next steps

- **Run the full eval suite**: `!make eval` (writes `reports/eval_summary.json` with bootstrap CIs).
- **Try the LLM synthesis preset (ADR 0011)**: set `BIDMATE_SYNTHESIS_BACKEND=anthropic` and `ANTHROPIC_API_KEY=...`, then `!python app.py --pipeline agentic_full_llm ...`.
- **Read the case study**: [`docs/portfolio-case-study.md`](https://github.com/hskim-solv/BidMate-DocAgent/blob/main/docs/portfolio-case-study.md) — five hypothesis→experiment→finding notes covering metadata-first retrieval, verifier/retry, hybrid BM25+dense, LLM synthesis, and embedding ablation.
- **Live demo (browser-only)**: [`docs/deployment.md`](https://github.com/hskim-solv/BidMate-DocAgent/blob/main/docs/deployment.md) covers Fly.io / Hugging Face Spaces / Railway recipes for the Streamlit UI.